# Elasticsearch + Jina AI — Unstructured Data Search

---
## Install Prerequisites

In [1]:
! pip install -q -U -r requirements.txt

---
## Provision Elastic Serverless (Terraform)

In [2]:
%%bash
terraform -chdir=terraform init -upgrade
terraform -chdir=terraform apply -auto-approve

Initializing the backend...
Initializing provider plugins...
- Finding elastic/elasticstack versions matching "~> 0.14"...
- Finding elastic/ec versions matching "~> 0.12"...
- Using previously-installed elastic/ec v0.12.4
- Using previously-installed elastic/elasticstack v0.14.3

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be created
  + resource "ec_elastics

---
## Create .env from Terraform Outputs

In [3]:
%%bash
cat > .env << EOF
ELASTIC_USERNAME=$(terraform -chdir=terraform output -raw elastic_username)
ELASTIC_PASSWORD=$(terraform -chdir=terraform output -raw elastic_password)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
JINA_API_KEY=$(terraform -chdir=terraform output -raw jina_api_key)
EMBED_ID=.jina-embeddings-v5-text-small
RERANK_ID=.jina-reranker-v3
EMBED_DIMENSIONS=1024
INDEX_NAME=manuals
INGEST_BATCH_SIZE=32
EOF

---
## Environment Setup & Index Creation

In [4]:
import os
import json
import asyncio
import re
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

load_dotenv(override=True)

ELASTIC_CLOUD_ID = os.environ["ELASTIC_CLOUD_ID"]
ELASTIC_USERNAME = os.environ["ELASTIC_USERNAME"]
ELASTIC_PASSWORD = os.environ["ELASTIC_PASSWORD"]
JINA_API_KEY     = os.environ["JINA_API_KEY"]
EMBED_ID         = os.environ.get("EMBED_ID", ".jina-embeddings-v5-text-small")
RERANK_ID        = os.environ.get("RERANK_ID", ".jina-reranker-v3")
EMBED_DIMENSIONS = int(os.environ.get("EMBED_DIMENSIONS", 1024))
INDEX_NAME       = os.environ.get("INDEX_NAME", "manuals")
BATCH_SIZE       = int(os.environ.get("INGEST_BATCH_SIZE", 32))

Path("data").mkdir(exist_ok=True)

es = Elasticsearch(
    cloud_id=ELASTIC_CLOUD_ID,
    basic_auth=(ELASTIC_USERNAME, ELASTIC_PASSWORD),
    request_timeout=60,
)

INDEX_MAPPING = {
    "mappings": {
        "properties": {
            "doc_id":             {"type": "keyword"},
            "guide_id":           {"type": "integer"},
            "brand":              {"type": "keyword"},
            "category":           {"type": "keyword"},
            "product":            {"type": "keyword"},
            "language":           {"type": "keyword"},
            "generation":         {"type": "integer"},
            "serial_range_start": {"type": "keyword"},
            "serial_range_end":   {"type": "keyword"},
            "source_url":         {"type": "keyword"},
            "content": {
                "type":         "semantic_text",
                "inference_id": EMBED_ID,
                "chunking_settings": {                                                                                                                                     
                    "strategy": "word",                                                                                                                                    
                    "max_chunk_size": 300,                                                                                                                                 
                    "overlap": 75                                                                                                                                          
                }   
            },
            "content_text": {"type": "text"},
        }
    }
}

es.options(ignore_status=[404]).indices.delete(index=INDEX_NAME)
es.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
print(f"Index '{INDEX_NAME}' created with semantic_text field (inference_id={EMBED_ID}).")

Index 'manuals' created with semantic_text field (inference_id=.jina-embeddings-v5-text-small).


---
## Create Dataset

In [5]:
GUIDE_URLS_PATH = Path("data/guide_urls.jsonl")
IFIXIT_BASE = "https://www.ifixit.com/api/2.0"

# Generation keywords → integer year proxy.
# Checked against both title AND category (which is the specific device model on iFixit).
GENERATION_PATTERNS = [
    # Explicit chip labels (newest first to avoid partial matches)
    (r"\bM3\b",         2024),
    (r"\bM2\b",         2023),
    (r"\bM1\b",         2021),
    # Explicit year keywords
    (r"\bLate 2020\b",  2020),
    (r"\bEarly 2020\b", 2020),
    (r"\bLate 2019\b",  2019),
    (r"\bMid 2019\b",   2019),
    (r"\bEarly 2019\b", 2019),
    (r"\b2019\b",       2019),
    (r"\b2018\b",       2018),
    (r"\b2017\b",       2017),
    (r"\b2016\b",       2016),
    (r"\b2015\b",       2015),
    (r"\b2014\b",       2014),
    (r"\b2013\b",       2013),
    (r"\b2012\b",       2012),
    (r"\b2011\b",       2011),
    (r"\b2010\b",       2010),
    (r"\b2009\b",       2009),
    (r"\b2008\b",       2008),
    (r"\b2007\b",       2007),
    (r"\b2006\b",       2006),
    # Chip-era fallbacks for older guides without explicit year
    (r"\bCore 2 Duo\b", 2007),
    (r"\bCore Duo\b",   2006),
    (r"\bG4\b",         2004),
    (r"\bG3\b",         2001),
    # Console generations
    (r"Series X\b",     2020),
    (r"Xbox One\b",     2013),
    (r"\bPS5\b",        2020),
    (r"\bPS4\b",        2013),
]

# Serial range registry keyed by (brand, generation)
SERIAL_RANGES = {
    ("Apple", 2024): ("C02Z", "C03Z"),
    ("Apple", 2023): ("C02Y", "C02Z"),
    ("Apple", 2021): ("C02X", "C02Y"),
    ("Apple", 2020): ("C02W", "C02X"),
    ("Apple", 2019): ("C02V", "C02W"),
    ("Apple", 2018): ("C02T", "C02V"),
    ("Apple", 2017): ("C02S", "C02T"),
    ("Apple", 2016): ("C02R", "C02S"),
    ("Apple", 2015): ("C02P", "C02R"),
    ("Apple", 2014): ("C02N", "C02P"),
    ("Apple", 2013): ("C02M", "C02N"),
    ("Apple", 2012): ("C02L", "C02M"),
    ("Apple", 2011): ("C02K", "C02L"),
    ("Apple", 2010): ("C02J", "C02K"),
    ("Apple", 2009): ("C02H", "C02J"),
    ("Apple", 2008): ("C02G", "C02H"),
    ("Apple", 2007): ("C02F", "C02G"),
    ("Apple", 2006): ("C02E", "C02F"),
    ("Apple", 2004): ("W85", "W86"),
    ("Apple", 2001): ("W80", "W81"),
}

# Brand hint lists — checked against combined "category + title" string
BRAND_HINTS = {
    "Apple": [
        "MacBook", "iMac", "iPad", "iPhone", "iPod",
        "iBook", "PowerBook", "Mac Pro", "Mac Mini",
        "Apple TV", "AirPort", " Mac ",
    ],
    "Samsung":    ["Samsung", "Galaxy"],
    "Google":     ["Google", "Pixel"],
    "OnePlus":    ["OnePlus"],
    "Sony":       ["Sony", "PlayStation", "PS4", "PS5", "PSP"],
    "Microsoft":  ["Microsoft", "Xbox", "Surface"],
    "Nintendo":   ["Nintendo", "Wii", "GameCube", "DS ", "3DS", "Switch"],
    "DeWalt":     ["DeWalt", "DEWALT"],
    "Makita":     ["Makita"],
    "Bosch":      ["Bosch"],
    "Honda":      ["Honda"],
    "Toyota":     ["Toyota"],
    "Ford":       ["Ford"],
    "Chevrolet":  ["Chevrolet", "Chevy"],
}

def parse_brand(category: str, title: str) -> str:
    text = f"{category} {title}"
    for brand, hints in BRAND_HINTS.items():
        if any(h in text for h in hints):
            return brand
    # Fallback: first word of category
    return category.split()[0] if category else "Unknown"

def parse_generation(title: str, category: str) -> int | None:
    """Check title first, then category (which is the specific device model on iFixit)."""
    for source in (title, category):
        for pattern, year in GENERATION_PATTERNS:
            if re.search(pattern, source, re.IGNORECASE):
                return year
    return None

def parse_serial_range(brand: str, generation: int | None) -> tuple[str | None, str | None]:
    if generation is not None:
        return SERIAL_RANGES.get((brand, generation), (None, None))
    return (None, None)


guide_metadata = []

if GUIDE_URLS_PATH.exists():
    with open(GUIDE_URLS_PATH) as f:
        guide_metadata = [json.loads(line) for line in f]
    print(f"Loaded {len(guide_metadata)} guides from cache.")
else:
    raw_guides = []
    with httpx.Client(timeout=30) as client:
        for offset in tqdm(range(0, 1200, 200), desc="Fetching iFixit pages"):
            resp = client.get(f"{IFIXIT_BASE}/guides", params={"limit": 200, "offset": offset})
            resp.raise_for_status()
            raw_guides.extend(resp.json())

    for g in raw_guides:
        title    = g.get("title", "")
        category = g.get("category", "")
        locale   = g.get("locale", "en")
        brand    = parse_brand(category, title)
        gen      = parse_generation(title, category)
        sr_start, sr_end = parse_serial_range(brand, gen)
        guide_metadata.append({
            "guideid":            g["guideid"],
            "title":              title,
            "category":           category,
            "url":                g.get("url", ""),
            "language":           locale,
            "brand":              brand,
            "generation":         gen,
            "serial_range_start": sr_start,
            "serial_range_end":   sr_end,
        })

    with open(GUIDE_URLS_PATH, "w") as f:
        for rec in guide_metadata:
            f.write(json.dumps(rec) + "\n")

    print(f"Collected {len(guide_metadata)} guides → saved to {GUIDE_URLS_PATH}")

# Quick stats
df_meta = pd.DataFrame(guide_metadata)
gen_filled = df_meta["generation"].notna().sum()
serial_filled = df_meta["serial_range_start"].notna().sum()
print(f"\nGuides with generation: {gen_filled}/{len(df_meta)}")
print(f"Guides with serial range: {serial_filled}/{len(df_meta)}")
print("\nTop brands:")
print(df_meta["brand"].value_counts().head(10))
print("\nGeneration distribution (top 10):")
print(df_meta["generation"].value_counts().head(10))

Loaded 1200 guides from cache.

Guides with generation: 626/1200
Guides with serial range: 626/1200

Top brands:
brand
Apple         1094
Sony            46
Microsoft       10
Nintendo         7
BlackBerry       5
Motorola         4
Palm             4
iFixit           2
Canon            2
Pleo             1
Name: count, dtype: int64

Generation distribution (top 10):
generation
2004.0    317
2001.0    107
2007.0    101
2006.0     77
2009.0     24
Name: count, dtype: int64


---
## Fetch Manual Content via Jina Reader

In [6]:
from tenacity import (
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential,
    wait_random,
    before_sleep_log,
    RetryError,
)

DOCS_PATH       = Path("data/docs.jsonl")
guides_to_fetch = guide_metadata


def _is_retryable(exc: BaseException) -> bool:
    if isinstance(exc, httpx.HTTPStatusError):
        # Retry on 429 (rate limit) or 5xx (server error); skip 4xx client errors
        return exc.response.status_code == 429 or exc.response.status_code >= 500
    return isinstance(exc, httpx.TimeoutException)


@retry(
    retry=retry_if_exception(_is_retryable),
    wait=wait_exponential(multiplier=1, min=1, max=10) + wait_random(0, 0.5),
    stop=stop_after_attempt(3),
    reraise=True,
)
async def fetch_with_retry(url: str, api_key: str, client: httpx.AsyncClient) -> dict:
    headers = {
        "Authorization":   f"Bearer {api_key}",
        "Accept":          "application/json",
        "X-Retain-Images": "none",
    }
    resp = await client.get(f"https://r.jina.ai/{url}", headers=headers, timeout=20)
    resp.raise_for_status()   # raises HTTPStatusError on 4xx/5xx — tenacity catches it
    return resp.json()


async def fetch_all_guides(guides: list[dict], api_key: str, concurrency: int = 10) -> list[dict]:
    sem  = asyncio.Semaphore(concurrency)
    docs = []

    async def process_one(guide: dict, client: httpx.AsyncClient, pbar) -> None:
        async with sem:
            try:
                result  = await fetch_with_retry(guide["url"], api_key, client)
                content = result.get("data", {}).get("content", "") or ""
                docs.append({
                    "doc_id":             str(guide["guideid"]),
                    "guide_id":           guide["guideid"],
                    "brand":              guide["brand"],
                    "category":           guide["category"],
                    "product":            guide["title"],
                    "language":           guide["language"],
                    "generation":         guide["generation"],
                    "serial_range_start": guide["serial_range_start"],
                    "serial_range_end":   guide["serial_range_end"],
                    "source_url":         guide["url"],
                    "content":            content,
                })
            except Exception as exc:
                print(f"  [WARN] guide {guide['guideid']}: {exc}")
            finally:
                pbar.update(1)

    async with httpx.AsyncClient(timeout=30) as client:
        with tqdm(total=len(guides), desc="Jina Reader") as pbar:
            await asyncio.gather(*[process_one(g, client, pbar) for g in guides])

    return docs

if DOCS_PATH.exists():
    with open(DOCS_PATH) as f:
        all_docs = [json.loads(line) for line in f]
    print(f"Loaded {len(all_docs)} docs from cache.")
else:
    all_docs = await fetch_all_guides(guides_to_fetch, JINA_API_KEY)
    with open(DOCS_PATH, "w") as f:
        for doc in all_docs:
            f.write(json.dumps(doc) + "\n")
    print(f"Fetched {len(all_docs)} docs → saved to {DOCS_PATH}")

df_docs = pd.DataFrame(all_docs)
print(f"\nTotal docs: {len(df_docs)}")
print(f"Avg content length: {df_docs['content'].str.len().mean():.0f} chars")

Loaded 1200 docs from cache.

Total docs: 1200
Avg content length: 21985 chars


---
## Bulk Indexing

In [7]:

def iter_actions(docs: list[dict]):
    for doc in tqdm(docs, desc="Indexing"):
        source = {**doc, "content_text": doc["content"]}
        yield {"_index": INDEX_NAME, "_id": doc["doc_id"], "_source": source}

successes, errors = bulk(
        es,
        iter_actions(all_docs),
        raise_on_error=False,
        chunk_size=BATCH_SIZE,
)
print(f"Indexed {successes} documents. Errors: {len(errors)}")
 
es.indices.refresh(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' now contains {es.count(index=INDEX_NAME)['count']} documents.")

Indexing:   0%|          | 0/1200 [00:00<?, ?it/s]

Indexed 1163 documents. Errors: 37
Index 'manuals' now contains 1163 documents.


---
## Shared Utility Functions


In [8]:
# ---------------------------------------------------------------------------
# Extract top passage text from a semantic_text hit
# ---------------------------------------------------------------------------

def top_chunk(hit: dict) -> str:
    """Return the highest-scoring passage from a semantic_text hit.

    With the retriever API, _source.content is the raw indexed string.
    With the legacy query/semantic API, it is a dict with inference.chunks.
    """
    content = hit.get("_source", {}).get("content", "")
    if isinstance(content, str):
        return content
    chunks = content.get("inference", {}).get("chunks", [])
    return chunks[0]["text"] if chunks else ""


# ---------------------------------------------------------------------------
# Display helper
# ---------------------------------------------------------------------------

def display_hits(hits: list[dict], fields: list[str] | None = None) -> pd.DataFrame:
    """Render ES hits as a DataFrame, pulling passage text from semantic_text chunks."""
    meta_fields = fields or ["brand", "category", "language"]
    rows = []
    for h in hits:
        src  = h.get("_source", {})
        text = top_chunk(h)
        row  = {"_score": round(h.get("_score") or 0, 4)}
        for f in meta_fields:
            row[f] = src.get(f, "")
        row["passage"] = (text[:160] + "…") if len(text) > 160 else text
        rows.append(row)
    return pd.DataFrame(rows)


print("Utility functions loaded.")

Utility functions loaded.


---
## Scenario 1 — Brand Isolation

In [9]:
QUERY_S1     = "battery replacement procedure"
TARGET_BRAND = "Sony"

# --- Unfiltered semantic search via retriever ---
resp_unfiltered = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query": {"semantic": {"field": "content", "query": QUERY_S1}},
        }
    },
    size=5
)
hits_unfiltered = resp_unfiltered["hits"]["hits"]

print("=== Unfiltered semantic search (may include multiple brands) ===")
display(display_hits(hits_unfiltered, ["brand", "category"]))

# --- Filtered to Sony only via retriever ---
resp_filtered = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query":  {"semantic": {"field": "content", "query": QUERY_S1}},
            "filter": {"term": {"brand": TARGET_BRAND}},
        }
    },
    size=5
)
hits_filtered = resp_filtered["hits"]["hits"]

print(f"\n=== Filtered semantic search (brand = '{TARGET_BRAND}' only) ===")
display(display_hits(hits_filtered, ["brand", "category"]))

brands_unfiltered = {h["_source"]["brand"] for h in hits_unfiltered}
brands_filtered   = {h["_source"]["brand"] for h in hits_filtered}
print(f"\nBrands in unfiltered result: {brands_unfiltered}")
print(f"Brands in filtered result:   {brands_filtered}")
assert brands_filtered == {TARGET_BRAND}, "FAIL: non-target brand leaked into filtered results"
print("PASS: Zero non-target brand results in filtered search.")

=== Unfiltered semantic search (may include multiple brands) ===


,_score,brand,category,passage
0,0.8402,Sony,PSP 2000,PSP 2000 Battery Replacement - iFixit Repair G...
1,0.8396,Apple,iPhone 3G,1. \n \n\n * If your display glass i...
2,0.8381,Apple,iPhone 3G,30 minutes - 1 hour\n\nModerate\n\nWhat you ne...
3,0.8327,Sony,PSP Go,PSP Go Battery Replacement - iFixit Repair Gui...
4,0.8302,Apple,iPhone 3G,No estimate\n\nDifficult\n\nIntroduction\n----...



=== Filtered semantic search (brand = 'Sony' only) ===


,_score,brand,category,passage
0,0.8402,Sony,PSP 2000,PSP 2000 Battery Replacement - iFixit Repair G...
1,0.8327,Sony,PSP Go,PSP Go Battery Replacement - iFixit Repair Gui...
2,0.8162,Sony,PSP Go,PSP Go Logic Board Replacement - iFixit Repair...
3,0.8110,Sony,PSP Go,PSP Go Display Replacement - iFixit Repair Gui...
4,0.8079,Sony,PSP Go,PSP Go Directional Pad Replacement - iFixit Re...



Brands in unfiltered result: {'Apple', 'Sony'}
Brands in filtered result:   {'Sony'}
PASS: Zero non-target brand results in filtered search.


---
## Scenario 2 — Fault Diagnosis

In [10]:
QUERY_S2 = "machine makes grinding noise after startup"

# --- BM25 only ---
resp_bm25 = es.search(
    index=INDEX_NAME,
    query={"match": {"content_text": QUERY_S2}},
    size=5
)
hits_bm25 = resp_bm25["hits"]["hits"]

print("=== Lexical Search ===")
display(display_hits(hits_bm25[:5], ["brand", "product"]))
print(f"  → BM25 found {len(hits_bm25)} results via exact keyword matching.")

# --- Semantic only via retriever ---
resp_sem = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query": {"semantic": {"field": "content", "query": QUERY_S2}},
        }
    },
    size=20,
)
hits_sem = resp_sem["hits"]["hits"]

print("\n=== Semantic Search ===")
display(display_hits(hits_sem[:5], ["brand", "product"]))

bm25_ids = {h["_id"] for h in hits_bm25[:5]}
sem_ids  = {h["_id"] for h in hits_sem[:5]}
overlap  = bm25_ids & sem_ids
print(f"  → {len(overlap)}/5 top results overlap with BM25 — semantic finds conceptually related docs that BM25 misses.")

# --- Semantic + Reranker v3 ---
resp_rerank = es.search(
    index=INDEX_NAME,
    retriever= {
        "text_similarity_reranker": {
            "retriever": {
                "standard": {
                    "query": {"semantic": {"field": "content", "query": QUERY_S2}},
                }
            },
            "field": "content",
            "inference_id": RERANK_ID,
            "inference_text": QUERY_S2,
            "rank_window_size": 5
        }
    }
)

print("\n=== Semantic Search + Reranker ===")
hits_rerank = resp_rerank["hits"]["hits"]
display(display_hits(hits_rerank[:5], ["brand", "product"]))

sem_rank = next((i for i, h in enumerate(hits_sem[:20]) if h["_id"] == hits_rerank[0]["_id"]), None)
if sem_rank is not None:
    print(f"  → Reranker promoted '{hits_rerank[0]['_source']['product']}' to #1 (was #{sem_rank + 1} in semantic).")
else:
    print(f"  → Reranker ranked '{hits_rerank[0]['_source']['product']}' #1.")

=== Lexical Search ===


,_score,brand,product,passage
0,14.2025,Apple,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1...","MacBook Pro 15"" Core 2 Duo Models A1226 and A1..."
1,13.2928,Apple,"MacBook Pro 15"" Unibody Late 2008 and Early 20...","MacBook Pro 15"" Unibody Late 2008 and Early 20..."
2,10.4977,Apple,Mac mini Model A1283 Terabyte Drive Replacement,30 minutes - 1 hour\n\nVery difficult\n\nIntro...
3,10.3664,Apple,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1...","MacBook Pro 15"" Core 2 Duo Models A1226 and A1..."
4,9.1325,Apple,"MacBook Pro 15"" Core Duo Model A1150 Right Fan...","MacBook Pro 15"" Core Duo Model A1150 Right Fan..."


  → BM25 found 5 results via exact keyword matching.

=== Semantic Search ===


,_score,brand,product,passage
0,0.7524,Apple,Mac mini Model A1176 Hard Drive Replacement,Mac mini Model A1176 Hard Drive Replacement - ...
1,0.7435,Apple,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1...","MacBook Pro 15"" Core 2 Duo Models A1226 and A1..."
2,0.7408,Apple,"iMac Intel 20"" EMC 2266 Hard Drive Replacement","iMac Intel 20"" EMC 2266 Hard Drive Replacement..."
3,0.7368,Sony,PlayStation 2 Teardown,PlayStation 2 Teardown - iFixit\n=============...
4,0.7239,Apple,PowerBook G3 Pismo Sound Card Replacement,PowerBook G3 Pismo Sound Card Replacement - iF...


  → 1/5 top results overlap with BM25 — semantic finds conceptually related docs that BM25 misses.

=== Semantic Search + Reranker ===


,_score,brand,product,passage
0,0.9516,Sony,PlayStation 2 Teardown,PlayStation 2 Teardown - iFixit\n=============...
1,0.9208,Apple,PowerBook G3 Pismo Sound Card Replacement,PowerBook G3 Pismo Sound Card Replacement - iF...
2,0.8811,Apple,Mac mini Model A1176 Hard Drive Replacement,Mac mini Model A1176 Hard Drive Replacement - ...
3,0.8761,Apple,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1...","MacBook Pro 15"" Core 2 Duo Models A1226 and A1..."
4,0.8709,Apple,"iMac Intel 20"" EMC 2266 Hard Drive Replacement","iMac Intel 20"" EMC 2266 Hard Drive Replacement..."


  → Reranker promoted 'PlayStation 2 Teardown' to #1 (was #4 in semantic).


---
## Scenario 3 — Version-Aware Search

In [11]:
QUERY_S3          = "battery replacement steps"
TECHNICIAN_SERIAL = "C02FX123"   # Falls in 2007 range: C02F–C02G

# --- Without serial filter (multiple generations returned) ---
resp_no_filter = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query": {"semantic": {"field": "content", "query": QUERY_S3}},
            "filter": {"exists": {"field": "generation"}},
        }
    },
    size=10
)
hits_no_filter = resp_no_filter["hits"]["hits"]

print("=== No serial filter (multiple generations returned) ===")
display(display_hits(hits_no_filter, ["brand", "product", "generation", "serial_range_start", "serial_range_end"]))

# --- Serial-scoped ---
resp_serial = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query": {"semantic": {"field": "content", "query": QUERY_S3}},
            "filter": {
                "bool": {
                    "must": [
                        {"range": {"serial_range_start": {"lte": TECHNICIAN_SERIAL}}},
                        {"range": {"serial_range_end":   {"gte": TECHNICIAN_SERIAL}}},
                    ]
                }
            },
        }
    },
    size=5
)
hits_serial = resp_serial["hits"]["hits"]

print(f"\n=== Serial-scoped (serial={TECHNICIAN_SERIAL}, 2007 Core 2 Duo range) ===")
if hits_serial:
    display(display_hits(hits_serial, ["brand", "product", "generation", "serial_range_start", "serial_range_end"]))
    generations = {h["_source"]["generation"] for h in hits_serial}
    assert generations == {2007}, f"FAIL: unexpected generations {generations}"
    print("PASS: Serial-scoped search returns only the 2007 generation.")
else:
    print("No results — check that 2007 MacBook guides were ingested.")

=== No serial filter (multiple generations returned) ===


,_score,brand,product,generation,serial_range_start,serial_range_end,passage
0,0.8263,Apple,PowerBook G3 Pismo PRAM Battery Replacement,2001,W80,W81,PowerBook G3 Pismo PRAM Battery Replacement - ...
1,0.8161,Apple,"PowerBook G4 Aluminum 15"" 1.67 GHz Left Ambien...",2004,W85,W86,"PowerBook G4 Aluminum 15"" 1.67 GHz Left Ambien..."
2,0.8132,Apple,"PowerBook G4 Aluminum 17"" 1-1.67 GHz Battery R...",2004,W85,W86,"PowerBook G4 Aluminum 17"" 1-1.67 GHz Battery R..."
3,0.8130,Apple,"PowerBook G4 Aluminum 15"" 1.67 GHz PRAM Batter...",2004,W85,W86,"PowerBook G4 Aluminum 15"" 1.67 GHz PRAM Batter..."
4,0.8098,Apple,PowerBook G3 Wallstreet PRAM Battery Replacement,2001,W80,W81,PowerBook G3 Wallstreet PRAM Battery Replaceme...
5,0.8081,Apple,"PowerBook G4 Aluminum 15"" 1.5-1.67 GHz Battery...",2004,W85,W86,"PowerBook G4 Aluminum 15"" 1.5-1.67 GHz Battery..."
6,0.8061,Apple,PowerBook G3 Lombard PRAM Battery Replacement,2001,W80,W81,PowerBook G3 Lombard PRAM Battery Replacement ...
7,0.8056,Apple,"PowerBook G4 Aluminum 15"" 1-1.5 GHz Battery Re...",2004,W85,W86,"PowerBook G4 Aluminum 15"" 1-1.5 GHz Battery Re..."
8,0.8035,Apple,"PowerBook G4 Aluminum 17"" 1.67 GHz (High-Res) ...",2004,W85,W86,"PowerBook G4 Aluminum 17"" 1.67 GHz (High-Res) ..."
9,0.8035,Apple,MacBook Core Duo PRAM Battery Replacement,2006,C02E,C02F,MacBook Core Duo PRAM Battery Replacement - iF...



=== Serial-scoped (serial=C02FX123, 2007 Core 2 Duo range) ===


,_score,brand,product,generation,serial_range_start,serial_range_end,passage
0,0.7947,Apple,"MacBook Pro 15"" Core 2 Duo Model A1211 Battery...",2007,C02F,C02G,"MacBook Pro 15"" Core 2 Duo Model A1211 Battery..."
1,0.7895,Apple,MacBook Core 2 Duo Battery Replacement,2007,C02F,C02G,MacBook Core 2 Duo Battery Replacement - iFixi...
2,0.7881,Apple,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1...",2007,C02F,C02G,"MacBook Pro 15"" Core 2 Duo Models A1226 and A1..."
3,0.7851,Apple,"MacBook Pro 15"" Core 2 Duo Model A1211 Battery...",2007,C02F,C02G,* Fix Your Stuff\n\n[Repair Guides Learn how...
4,0.7819,Apple,"MacBook Pro 15"" Core 2 Duo Model A1211 PRAM Ba...",2007,C02F,C02G,"MacBook Pro 15"" Core 2 Duo Model A1211 PRAM Ba..."


PASS: Serial-scoped search returns only the 2007 generation.


---
## Scenario 4 — Cross-Lingual Retrieval


In [12]:
QUERY_S4_FR = "l'écran ne répond plus au toucher"

# --- BM25 with French query (few/no relevant results — zero English cognates) ---
resp_bm25_fr = es.search(
    index=INDEX_NAME,
    query={"match": {"content_text": QUERY_S4_FR}},
    size=5,
)
hits_bm25_fr = resp_bm25_fr["hits"]["hits"]

print("=== BM25 with French query ===")
if hits_bm25_fr:
    print(f"BM25 returned {len(hits_bm25_fr)} result(s) — but scores are low, results are irrelevant (no content-word overlap with French query):")
    display(display_hits(hits_bm25_fr, ["language", "brand", "product"]))
else:
    print("BM25 returned 0 results — no English tokens match the French query.")

# --- Semantic with French query (cross-lingual) via retriever ---
resp_sem_fr = es.search(
    index=INDEX_NAME,
    retriever={
        "standard": {
            "query": {"semantic": {"field": "content", "query": QUERY_S4_FR}},
        }
    },
    size=5,
    aggs={"by_language": {"terms": {"field": "language", "size": 10}}},
)
hits_sem_fr  = resp_sem_fr["hits"]["hits"]
lang_buckets = resp_sem_fr["aggregations"]["by_language"]["buckets"]

print("\n=== Cross-lingual semantic (French query → English results) — top 5 ===")
display(display_hits(hits_sem_fr, ["language", "brand", "product"]))

# --- Semantic + Reranker v3 ---
resp_rerank_fr = es.search(
    index=INDEX_NAME,
    retriever= {
        "text_similarity_reranker": {
            "retriever": {
                "standard": {
                    "query": {"semantic": {"field": "content", "query": QUERY_S4_FR}},
                }
            },
            "field": "content",
            "inference_id": RERANK_ID,
            "inference_text": QUERY_S4_FR,
            "rank_window_size": 5,
        }
    },
    source_excludes=["content.inference.chunks.embeddings"],
)
hits_rerank_fr = resp_rerank_fr["hits"]["hits"]

print("\n=== Semantic + Reranker v3 (French query, top 5) ===")
display(display_hits(hits_rerank_fr[:5], ["language", "brand", "product"]))

sem_rank = next((i for i, h in enumerate(hits_sem_fr[:20]) if h["_id"] == hits_rerank_fr[0]["_id"]), None)
if sem_rank is not None:
    print(f"  → Reranker promoted '{hits_rerank_fr[0]['_source']['product']}' to #1 (was #{sem_rank + 1} in semantic).")
else:
    print(f"  → Reranker ranked '{hits_rerank_fr[0]['_source']['product']}' #1.")

assert len(hits_sem_fr) > 0, "FAIL: Semantic search returned no results for French query."
print("PASS: Semantic search bridges the language gap that BM25 cannot cross.")

=== BM25 with French query ===
BM25 returned 5 result(s) — but scores are low, results are irrelevant (no content-word overlap with French query):


,_score,language,brand,product,passage
0,12.9910,en,Apple,MacBook Unibody Model A1278 Fan Replacement,MacBook Unibody Model A1278 Fan Replacement - ...
1,6.3296,en,Apple,"iMac Intel 20"" EMC 2266 Teardown",Introduction\n------------\n\n[Go to step 1](h...
2,5.5428,en,Apple,MacBook Air Models A1237 and A1304 Teardown,Introduction\n------------\n\n[Go to step 1](h...
3,3.9723,en,Apple,"iMac Intel 20"" EMC 2133 and 2210 Heat Sink Re...","iMac Intel 20"" EMC 2133 and 2210 Heat Sink Rep..."
4,0.1315,en,Apple,iPod Shuffle 1st Generation Hold Switch Replac...,* Fix Your Stuff\n\n[Repair Guides Learn how...



=== Cross-lingual semantic (French query → English results) — top 5 ===


,_score,language,brand,product,passage
0,0.7927,en,Apple,iPod Touch 2nd Generation Front Panel Replacement,iPod Touch 2nd Generation Front Panel Replacem...
1,0.7797,en,T-Mobile,T-Mobile G1 Teardown,T-Mobile G1 Teardown - iFixit\n===============...
2,0.7792,en,Apple,iPhone 3G Front Panel Replacement,iPhone 3G Front Panel Replacement - iFixit Rep...
3,0.7594,en,Apple,iPhone 1st Generation Display Assembly Replace...,iPhone 1st Generation Display Assembly Replace...
4,0.7477,en,Apple,"iMac Intel 20"" EMC 2133 and 2210 Display Panel...","iMac Intel 20"" EMC 2133 and 2210 Display Panel..."



=== Semantic + Reranker v3 (French query, top 5) ===


,_score,language,brand,product,passage
0,0.9863,en,Apple,iPod Touch 2nd Generation Front Panel Replacement,iPod Touch 2nd Generation Front Panel Replacem...
1,0.9551,en,Apple,iPhone 1st Generation Display Assembly Replace...,iPhone 1st Generation Display Assembly Replace...
2,0.9180,en,Apple,"iMac Intel 20"" EMC 2133 and 2210 Display Panel...","iMac Intel 20"" EMC 2133 and 2210 Display Panel..."
3,0.9067,en,Apple,iPhone 3G Front Panel Replacement,iPhone 3G Front Panel Replacement - iFixit Rep...
4,0.9057,en,T-Mobile,T-Mobile G1 Teardown,T-Mobile G1 Teardown - iFixit\n===============...


  → Reranker promoted 'iPod Touch 2nd Generation Front Panel Replacement' to #1 (was #1 in semantic).
PASS: Semantic search bridges the language gap that BM25 cannot cross.


---
## Teardown — Destroy Elastic Environment

In [13]:
%%bash
terraform -chdir=terraform destroy -auto-approve
rm -f .env

ec_elasticsearch_project.demo_project: Refreshing state... [id=eee00318e3f348c8a9641006c7e60a0b]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be destroyed
  - resource "ec_elasticsearch_project" "demo_project" {
      - alias         = "demoproject" -> null
      - cloud_id      = "demo_project:dXMtY2VudHJhbDEuZ2NwLmVsYXN0aWMuY2xvdWQkZWVlMDAzMThlM2YzNDhjOGE5NjQxMDA2YzdlNjBhMGIuZXMkZWVlMDAzMThlM2YzNDhjOGE5NjQxMDA2YzdlNjBhMGIua2I=" -> null
      - credentials   = {
          - password = (sensitive value) -> null
          - username = "admin" -> null
        } -> null
      - endpoints     = {
          - elasticsearch = "https://demoproject-eee003.es.us-central1.gcp.elastic.cloud" -> null
          - kibana        = "https://demoproject-eee003.kb.us-central1.gcp.elastic.cloud" -> null
  